# Top-1000 Stocks by Trading Volume — Paper Universe Replication

Replicates the initial stock pool from:
> *Stock portfolio selection using learning-to-rank algorithms with news sentiment*

**Method (paper §3.1):** Select the top 1,000 US common stocks by average daily share volume over
the paper window (2003-01-01 to 2014-12-31), restricted to ordinary common shares listed on
NYSE, AMEX, or NASDAQ.

**Source:** WRDS CRSP — `crsp.dsf` (daily security file) joined to `crsp.msenames` (security name history).

**Filters applied (matching the paper's Bloomberg-based selection as closely as possible via CRSP):**
- `shrcd IN (10, 11)` — ordinary common shares only (no ETFs, ADRs, REITs, etc.)
- `exchcd IN (1, 2, 3)` — NYSE (1), AMEX/ARCA (2), NASDAQ (3)
- Non-null volume observations only

**Ranking:** Top 1,000 by `AVG(vol)` (average daily share volume) over the full window.

**Note:** This is the *market-side* candidate universe only. The paper's final stock pool also applies a
TRNA news-coverage filter (≥1 article/week on average). That filter is NOT applied here.

---
Credentials are read from `../.env` (`WRDS_USERNAME`, `WRDS_PASSWORD`).

In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import plotly.express as px
import wrds
from dotenv import load_dotenv

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")

## Parameters

In [ ]:
START_DATE = "2003-01-01"
END_DATE   = "2014-12-31"
CANDIDATE_COUNT = 1_000

# shrcd 10 = ordinary common share, 11 = ordinary common share (foreign incorporated)
SHARE_CODES = [10, 11]
# exchcd 1 = NYSE, 2 = AMEX/ARCA, 3 = NASDAQ
EXCHANGE_CODES = [1, 2, 3]

OUTPUT_DIR  = PROJECT_ROOT / "data" / "raw" / "market"
OUTPUT_CSV  = OUTPUT_DIR / "crsp_top_volume_universe.csv"
OUTPUT_MANIFEST = OUTPUT_DIR / "crsp_top_volume_universe_manifest.json"

print(f"Window : {START_DATE} → {END_DATE}")
print(f"Top N   : {CANDIDATE_COUNT:,}")
print(f"shrcd   : {SHARE_CODES}")
print(f"exchcd  : {EXCHANGE_CODES}")
print(f"Output  : {OUTPUT_CSV}")

## Connect to WRDS

In [ ]:
WRDS_USERNAME = os.environ.get("WRDS_USERNAME")
WRDS_PASSWORD = os.environ.get("WRDS_PASSWORD")

if not WRDS_USERNAME or not WRDS_PASSWORD:
    raise EnvironmentError(
        "WRDS_USERNAME and WRDS_PASSWORD must be set — add them to ../.env or export them."
    )

db = wrds.Connection(wrds_username=WRDS_USERNAME, wrds_password=WRDS_PASSWORD)
print("WRDS connection established.")

## Build the SQL Query

Three CTEs:
1. **`eligible_daily`** — filter `crsp.dsf` to the paper window, join to `crsp.msenames` for share/exchange codes.
2. **`volume_rank`** — aggregate per PERMNO: average volume, dollar volume, price, shares outstanding.
3. **`top_candidates`** — take the top N by `avg_volume`.
4. **`latest_names`** — pull the most recent name record for each top-N PERMNO.

Final SELECT attaches name columns and assigns a `volume_rank` integer 1–N.

In [ ]:
def _sql_list(values: list[int]) -> str:
    return ", ".join(str(v) for v in values)


SQL = f"""
with eligible_daily as (
    select
        d.permno,
        d.date,
        abs(d.prc)      as price,
        d.vol,
        d.ret,
        d.shrout
    from crsp.dsf as d
    join crsp.msenames as n
      on d.permno  = n.permno
     and d.date between n.namedt and n.nameendt
    where d.date between '{START_DATE}' and '{END_DATE}'
      and n.shrcd  in ({_sql_list(SHARE_CODES)})
      and n.exchcd in ({_sql_list(EXCHANGE_CODES)})
      and d.vol is not null
),
volume_rank as (
    select
        permno,
        count(*)           as trading_days,
        min(date)          as first_trade_date,
        max(date)          as last_trade_date,
        avg(vol)           as avg_volume,
        avg(price * vol)   as avg_dollar_volume,
        avg(price)         as avg_abs_price,
        avg(shrout)        as avg_shares_outstanding
    from eligible_daily
    group by permno
),
top_candidates as (
    select *
    from volume_rank
    order by avg_volume desc
    limit {CANDIDATE_COUNT}
),
latest_names as (
    select distinct on (n.permno)
        n.permno,
        n.permco,
        n.ticker,
        n.comnam,
        n.shrcd,
        n.exchcd,
        n.namedt,
        n.nameendt
    from crsp.msenames as n
    join top_candidates as t
      on n.permno = t.permno
    where n.namedt   <= '{END_DATE}'
      and n.nameendt >= '{START_DATE}'
    order by n.permno, n.nameendt desc, n.namedt desc
)
select
    row_number() over (order by t.avg_volume desc) as volume_rank,
    t.permno,
    n.permco,
    n.ticker,
    n.comnam,
    n.shrcd,
    n.exchcd,
    t.trading_days,
    t.first_trade_date,
    t.last_trade_date,
    t.avg_volume,
    t.avg_dollar_volume,
    t.avg_abs_price,
    t.avg_shares_outstanding,
    n.namedt   as latest_name_start,
    n.nameendt as latest_name_end
from top_candidates as t
left join latest_names as n
  on t.permno = n.permno
order by t.avg_volume desc
"""

print(SQL)

## Run Query Against WRDS

This query scans the full 2003–2014 CRSP daily file (~25M rows) server-side.
Expect it to take 1–3 minutes.

In [ ]:
DATE_COLS = ["first_trade_date", "last_trade_date", "latest_name_start", "latest_name_end"]

try:
    universe = db.raw_sql(SQL, date_cols=DATE_COLS)
finally:
    db.close()

print(f"Rows returned: {len(universe):,}")
universe.head(10)

## Validation Checks

In [ ]:
checks = {
    "has_1000_rows":           len(universe) == CANDIDATE_COUNT,
    "volume_rank_is_unique":   universe["volume_rank"].is_unique,
    "permno_is_unique":        universe["permno"].is_unique,
    "avg_volume_descending":   universe["avg_volume"].is_monotonic_decreasing,
    "share_codes_in_filter":   set(universe["shrcd"].dropna().astype(int)).issubset(set(SHARE_CODES)),
    "exchange_codes_in_filter": set(universe["exchcd"].dropna().astype(int)).issubset(set(EXCHANGE_CODES)),
}

validation = pd.Series(checks, name="passed").to_frame()
print(validation.to_string())

> **Note on share/exchange code checks:** A small number of rows may carry `NaN` for `shrcd`/`exchcd`
> when a PERMNO lacks a matching name record at the exact boundary dates. The underlying CRSP filter
> is applied server-side in `eligible_daily`; the check above is a post-hoc sanity check on the
> joined name columns only.

## Derived Columns

In [ ]:
universe["avg_volume_millions"]       = universe["avg_volume"]       / 1_000_000
universe["avg_dollar_volume_billions"] = universe["avg_dollar_volume"] / 1_000_000_000

display_cols = [
    "volume_rank", "permno", "ticker", "comnam",
    "shrcd", "exchcd", "trading_days",
    "avg_volume_millions", "avg_dollar_volume_billions",
    "first_trade_date", "last_trade_date",
]
universe[display_cols].head(20).style.format(
    {
        "avg_volume_millions":        "{:,.2f}",
        "avg_dollar_volume_billions": "{:,.2f}",
        "first_trade_date":  lambda v: v.strftime("%Y-%m-%d") if pd.notna(v) else "",
        "last_trade_date":   lambda v: v.strftime("%Y-%m-%d") if pd.notna(v) else "",
    }
)

## Top-20 Bar Chart

In [ ]:
top20 = universe.head(20).copy()
plot_data = top20.sort_values("avg_volume_millions", ascending=True).copy()
plot_data["label"] = plot_data["ticker"] + " — " + plot_data["comnam"].str.title()

fig = px.bar(
    plot_data,
    x="avg_volume_millions",
    y="label",
    orientation="h",
    title="Top 20 CRSP Common Stocks by Average Daily Share Volume, 2003–2014",
    labels={
        "avg_volume_millions": "Average daily volume (millions of shares)",
        "label": "",
    },
    hover_data={
        "label":                    False,
        "ticker":                   True,
        "comnam":                   True,
        "permno":                   True,
        "volume_rank":              True,
        "trading_days":             ":,",
        "avg_volume_millions":       ":,.2f",
        "avg_dollar_volume_billions": ":,.2f",
        "first_trade_date":          "|%Y-%m-%d",
        "last_trade_date":           "|%Y-%m-%d",
    },
    color_discrete_sequence=["#4C78A8"],
)
fig.update_layout(
    height=700,
    hovermode="closest",
    yaxis={"categoryorder": "array", "categoryarray": plot_data["label"].tolist()},
)
fig.show()

## Volume Distribution Across the Full 1,000

In [ ]:
fig = px.histogram(
    universe,
    x="avg_volume_millions",
    nbins=50,
    title="Distribution of Average Daily Share Volume — Top 1,000 CRSP Stocks, 2003–2014",
    labels={"avg_volume_millions": "Average daily volume (millions of shares)"},
    color_discrete_sequence=["#4C78A8"],
)
fig.update_layout(height=450, bargap=0.05)
fig.show()

print(universe["avg_volume_millions"].describe().to_string())

## Spot-Check Key Tickers

In [ ]:
SPOT_TICKERS = ["AAPL", "MSFT", "GE", "XOM", "SPY", "C", "BAC", "INTC", "CSCO", "JPM"]

spot = universe[universe["ticker"].isin(SPOT_TICKERS)][
    ["volume_rank", "ticker", "comnam", "avg_volume_millions", "trading_days"]
].sort_values("volume_rank")

print(f"Found {len(spot)}/{len(SPOT_TICKERS)} spot-check tickers in the top-1000 universe:")
print(spot.to_string(index=False))

## Exchange and Share-Code Breakdown

In [ ]:
EXCHCD_LABELS = {1: "NYSE", 2: "AMEX/ARCA", 3: "NASDAQ"}
SHRCD_LABELS  = {10: "Ordinary (domestic)", 11: "Ordinary (foreign-inc.)"}

exch_counts = (
    universe["exchcd"]
    .dropna()
    .astype(int)
    .map(EXCHCD_LABELS)
    .value_counts()
    .rename("count")
    .to_frame()
)
print("Exchange breakdown:")
print(exch_counts.to_string())

shr_counts = (
    universe["shrcd"]
    .dropna()
    .astype(int)
    .map(SHRCD_LABELS)
    .value_counts()
    .rename("count")
    .to_frame()
)
print("\nShare-code breakdown:")
print(shr_counts.to_string())

## Save Results

Two copies are written:
- `data/raw/market/crsp_top_volume_universe.csv` — canonical research artifact (gitignored; regenerate from this notebook)
- `app_data/crsp_top_volume_universe.csv` — **git-tracked** copy; load this from any other notebook:

```python
universe = pd.read_csv(PROJECT_ROOT / "app_data" / "crsp_top_volume_universe.csv",
                       parse_dates=["first_trade_date", "last_trade_date"])
```

In [ ]:
import shutil

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

universe.to_csv(OUTPUT_CSV, index=False)

# Keep the git-tracked copy in sync so other notebooks can load it without WRDS.
APP_DATA_CSV = PROJECT_ROOT / "app_data" / "crsp_top_volume_universe.csv"
shutil.copy2(OUTPUT_CSV, APP_DATA_CSV)

manifest = {
    "created_at":      datetime.now(timezone.utc).isoformat(),
    "source":          "wrds.crsp.dsf joined to wrds.crsp.msenames",
    "start":           START_DATE,
    "end":             END_DATE,
    "candidate_count": CANDIDATE_COUNT,
    "share_codes":     SHARE_CODES,
    "exchange_codes":  EXCHANGE_CODES,
    "rows":            int(len(universe)),
    "columns":         list(universe.columns),
    "output_file":     str(OUTPUT_CSV.relative_to(PROJECT_ROOT)),
    "tracked_copy":    str(APP_DATA_CSV.relative_to(PROJECT_ROOT)),
    "ranking_rule":    "Top securities by average CRSP daily share volume over the configured date range.",
    "note":            (
        "This is the market-side candidate universe only. "
        "The paper's final universe also filters by TRNA news coverage (≥1 article/week on average)."
    ),
}
OUTPUT_MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Saved {len(universe):,} rows → {OUTPUT_CSV}")
print(f"Synced git-tracked copy  → {APP_DATA_CSV}")
print(f"Saved manifest           → {OUTPUT_MANIFEST}")
print()
print("To load in another notebook:")
print("  universe = pd.read_csv(PROJECT_ROOT / 'app_data' / 'crsp_top_volume_universe.csv',")
print("                         parse_dates=['first_trade_date', 'last_trade_date'])")